# 04_206 · Qwen en cascada y jerárquico multitarea · cuatro daños

Compara el Qwen plano de `04_205` con dos estructuras construidas sobre su adaptador terminado: una cascada logística y una cabeza neuronal jerárquica multitarea. Este cuaderno **no debe ejecutarse hasta que `04_205` termine**. No modifica su adaptador ni realiza un segundo fine-tuning end-to-end.

El arranque híbrido usa el workspace en local y, en Colab, código fijado de GitHub con artefactos mínimos persistentes en Google Drive.

In [ ]:
from pathlib import Path
from importlib.util import find_spec
import json, os, shutil, subprocess, sys
import pandas as pd
from IPython.display import display, Markdown, Image
REPO_URL = 'https://github.com/lkoc/Trabajo_PLN-MIA-Grupo4.git'
GIT_COMMIT = '131016751517333284ee24559d51d2bdebebc960'
PROJECT_NAME, DRIVE_BUNDLE_NAME = 'Trabajo_PLN-MIA-Grupo4', 'PLN_colab_04_artifacts'
def _bootstrap_04_20x():
    in_colab = find_spec('google.colab') is not None
    if not in_colab:
        start = Path.cwd().resolve()
        root = next((p for p in (start, *start.parents) if (p / 'scripts_auxiliares').is_dir()), None)
        if root is None: raise FileNotFoundError('No se encontró la raíz local del proyecto.')
        return root, False, root, 'working-tree-local'
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    artifacts = Path('/content/drive/MyDrive') / DRIVE_BUNDLE_NAME
    manifest_path = artifacts / 'MANIFIESTO_ARTEFACTOS_04_20X.json'
    if not manifest_path.is_file(): raise FileNotFoundError('Falta el bundle de Drive; ejecute sincronizar_04_20x_google_drive.ps1 en Windows.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8-sig'))
    missing = [r['path'] for r in manifest['files'] if not (artifacts / r['path']).is_file()]
    if missing: raise FileNotFoundError('Bundle incompleto en Drive:\n' + '\n'.join(missing))
    root = Path('/content') / PROJECT_NAME
    if not (root / '.git').is_dir():
        if root.exists(): raise RuntimeError(f'Ruta no administrada ya existente: {root}')
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(root)], check=True)
    current = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
    if current != GIT_COMMIT:
        subprocess.run(['git', '-C', str(root), 'fetch', '--depth', '1', 'origin', GIT_COMMIT], check=True)
    subprocess.run(['git', '-C', str(root), 'sparse-checkout', 'set', 'scripts_auxiliares'], check=True)
    subprocess.run(['git', '-C', str(root), 'checkout', '--detach', GIT_COMMIT], check=True)
    for name in ('datos', 'modelos', 'resultados'):
        local, target = root / name, artifacts / name
        target.mkdir(parents=True, exist_ok=True)
        if local.is_symlink() and local.resolve() == target.resolve(): continue
        if local.is_symlink(): local.unlink()
        elif local.exists(): raise RuntimeError(f'Ruta de artefactos no administrada: {local}')
        local.symlink_to(target, target_is_directory=True)
    return root, True, artifacts, GIT_COMMIT
ROOT, IN_COLAB, ARTIFACT_ROOT, CODE_VERSION = _bootstrap_04_20x()
if IN_COLAB:
    packages = {'transformers': 'transformers>=4.51,<6', 'sklearn': 'scikit-learn>=1.4', 'peft': 'peft>=0.15,<1'}
    missing_packages = [p for m, p in packages.items() if find_spec(m) is None]
    if missing_packages: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages], check=True)
os.environ['PLN_PROJECT_ROOT'], os.environ['PLN_ARTIFACT_ROOT'] = str(ROOT), str(ARTIFACT_ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from scripts_auxiliares import experimentos_qwen_jerarquico_4 as qh
from scripts_auxiliares import entrenar_qwen_acoso_amenaza as q4
print('Entorno:', 'Google Colab híbrido' if IN_COLAB else 'local')
print('Código:', ROOT, CODE_VERSION)
print('Artefactos persistentes:', ARTIFACT_ROOT)
print('Estado 04_205:', q4.resume_status())
print('Objetivos:', qh.TARGET_LABELS)

## 1. Contrato y requisito

La siguiente celda se detiene deliberadamente si `04_205` no generó `finetuning.json` y `best_adapter`. Cuando esté listo, verifica que Qwen y los experimentos jerárquicos compartan el mismo hash de dataset y los mismos splits.

In [ ]:
USE_EXPANDED_SAFE = True
context = qh.load_context(use_expanded_safe=USE_EXPANDED_SAFE)
display(context['qwen'])
display(context['expansion'])
assert context['qwen_audit']['dataset_sha256'] == context['frozen']['dataset_sha256']
assert context['expansion']['validation_or_test_videos_used'] is False

## 2. Representaciones Qwen congeladas

Se guardan los 21 logits de `04_205`: cuatro operativos, 14 finos auxiliares y tres flags. Son representaciones generadas únicamente a partir del texto; las etiquetas no son entradas. El cache registra hash del adaptador y de los IDs.

Con `USE_EXPANDED_SAFE=True`, Qwen debe inferir también los `SEGURO` adicionales permitidos. Esto puede tardar bastante en CPU, pero se realiza una sola vez y luego se reutiliza. Aunque el corpus tiene más de 110 mil `SEGURO`, train usa sólo aquellos cuyos videos no aparecen en validation/test.

In [ ]:
FORCE_FEATURES = False
features = qh.extract_features(context, force=FORCE_FEATURES)
display({split: values.shape for split, values in features.items()})

## 3. Dos diseños Qwen jerárquicos

La cascada entrena una puerta binaria con todos los negativos permitidos y cuatro cabezas condicionales con daños y negativos difíciles. El multitarea usa una capa compartida de 32 unidades, cabeza binaria y cuatro cabezas temáticas. La pérdida temática se enmascara en `SEGURO` adicional; sólo la cabeza binaria aprovecha esos chunks.

Ambos modelos fijan umbrales en validation y se comparan con las probabilidades calibradas del Qwen plano sobre exactamente el mismo test. La inferencia estadística usa bootstrap pareado por videos.

In [ ]:
FORCE = False
BOOTSTRAP_REPLICATES = 1_000
result = qh.run_experiment(
    force=FORCE,
    use_expanded_safe=USE_EXPANDED_SAFE,
    bootstrap_replicates=BOOTSTRAP_REPLICATES,
)
display(result['selection'])
display(result['paired_decisions_vs_qwen_flat'])

In [ ]:
comparison = pd.read_csv(qh.METRICS_DIR / 'comparacion.csv')
display(comparison)
display(pd.DataFrame(result['models']['qwen_frozen_joint']['training']['history']))
display(Image(filename=str(qh.FIGURES_DIR / 'comparacion_test.png')))

In [ ]:
display(Markdown(
    f'**Resultado:** `{qh.RESULT_PATH.relative_to(ROOT)}`  \n'
    f'**Informe:** `{qh.REPORT_PATH.relative_to(ROOT)}`  \n'
    f'**Modelos:** `{qh.MODEL_DIR.relative_to(ROOT)}`'
))

## Referencias (APA 7)

Cawley, G. C., & Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079–2107. https://www.jmlr.org/papers/v11/cawley10a.html

Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall/CRC.

Zhou, J., Ma, C., Long, D., Xu, G., Ding, N., Zhang, H., Xie, P., & Liu, G. (2020). Hierarchy-aware global model for hierarchical text classification. In *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics* (pp. 1106–1117). Association for Computational Linguistics. https://doi.org/10.18653/v1/2020.acl-main.104